# Visualización de los datos (10%)
En este notebook implementaremos visualizaciones bidimensionales usando:
- PCA (Principal Component Analysis)
- LDA (Linear Discriminant Analysis)
- ICA (Independent Component Analysis)
- t-SNE (t-Distributed Stochastic Neighbor Embedding)
- Isomap
- LLE (Locally Linear Embedding)

# Importaciones

In [ ]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")

JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval

## Preparación de datos

Separamos train de test para aplicar las técnicas correctamente

In [ ]:
# Extraer datos de test
y_test = df_test.Activity
X_test = DataFrames.select(df_test, Not([:subject, :Activity]))

println("Train: ", size(X_trainval))
println("Test: ", size(X_test))
println("Clases en train: ", unique(y_trainval))
println("Clases en test: ", unique(y_test))

## 1. PCA (Principal Component Analysis)

Se escalan los datos de entrenamiento y test con Min-Max, y después se aplica PCA para quedarnos con solo dos dimensiones. El PCA se ajusta empleando únicamente el conjunto de entrenamiento, y luego se usa esa misma transformación para proyectar los datos de test. Por último, se muestran ambas proyecciones en un plano 2D para ver cómo se distribuyen las clases tras la reducción de dimensionalidad.

In [ ]:
mach_scaler = machine(MyMinMaxScaler(), X_trainval)
fit!(mach_scaler)

Xtrain_scaled = MMI.transform(mach_scaler, X_trainval)
Xtest_scaled  = MMI.transform(mach_scaler, X_test)
println("Escalado hecho")


#PCA
pca_model = PCA(maxoutdim=2)  
mach_pca = machine(pca_model, Xtrain_scaled)
fit!(mach_pca)
Xtrain_pca = MMI.transform(mach_pca, Xtrain_scaled)
Xtest_pca = MMI.transform(mach_pca, Xtest_scaled)
println("PCA hecho")

plot_2d_projection(Xtrain_pca, y_trainval;
                   test2d=Xtest_pca, ytest=y_test,
                   title_str="PCA")

## 2. ICA (Independent Component Analysis)
Con los datos ya escalados, aplicamos ICA para reducir los datos a dos dimensiones independientes. Al igual que con PCA, ICA se ajusta únicamente con el conjunto de entrenamiento. Esta proyección se usa luego para transformar el conjunto de test, manteniendo la coherencia entre ambos. Al final se visualizan los datos proyectados para observar cómo se distribuyen las clases en este nuevo espacio de dos componentes.

In [ ]:
#ICA
ica_model = ICA(outdim=2)
mach_ica = machine(ica_model, Xtrain_scaled)
fit!(mach_ica)
Xtrain_ica = MMI.transform(mach_ica, Xtrain_scaled)
Xtest_ica = MMI.transform(mach_ica, Xtest_scaled)
println("ICA hecho")

plot_2d_projection(Xtrain_ica, y_trainval;
                   test2d=Xtest_ica, ytest=y_test,
                   title_str="ICA")

## 3. LDA (Linear Discriminant Analysis)
Aquí aplicamos LDA para proyectar los datos a un espacio bidimensional maximizando la separación entre clases. A diferencia de PCA o ICA, el modelo se entrena con los datos de entrenamiento junto con sus etiquetas, y luego la misma transformación se aplica al conjunto de test. Finalmente, se muestran las proyecciones para comparar cómo se agrupan las clases después de la reducción supervisada.

In [ ]:
#LDA
lda_model = LDA(outdim=2)
mach_lda = machine(lda_model, Xtrain_scaled, y_trainval)
fit!(mach_lda)
Xtrain_lda = MMI.transform(mach_lda, Xtrain_scaled)
Xtest_lda = MMI.transform(mach_lda, Xtest_scaled)
println("LDA hecho")    


plot_2d_projection(Xtrain_lda, y_trainval;
                   test2d=Xtest_lda, ytest=y_test,
                   title_str="LDA")

## 4. t-SNE (t-distributed Stochastic Neighbor Embedding)
Aquí empleamos t-SNE para obtener una proyección no lineal en dos dimensiones. Como esta técnica es computacionalmente costosa, primero se aplican 30 componentes de PCA y se usa un subconjunto del conjunto de entrenamiento para acelerar el cálculo. El objetivo es observar de manera visual cómo se agrupan las clases en el plano 2D resultante.

In [ ]:
# PCA previa a 30 dims
pca30 = PCA(maxoutdim=30)
mach_pca30 = machine(pca30, Xtrain_scaled)
fit!(mach_pca30)

Xtr_pca30 = MMI.transform(mach_pca30, Xtrain_scaled)
Xtr_pca30_mat = MMI.matrix(Xtr_pca30)

# Submuestreo
n = size(Xtr_pca30_mat, 1)
idx = randperm(n)[1:2000]
Xtr_pca30_small = Xtr_pca30_mat[idx, :]
y_small = y_trainval[idx]

# t-SNE directo (sin parámetros avanzados)
tsne_output = tsne(Xtr_pca30_small, 2)   # (N×2)


println("t-SNE hecho")

plot_2d_projection(tsne_output, y_small;
                   title_str="t-SNE")

## 5. Isomap

En esta parte se aplica Isomap para generar una proyección bidimensional basada en distancias geodésicas. Esta técnica intenta preservar la estructura global del espacio original, por lo que es útil para visualizar datos que puedan vivir en una variedad no lineal. Igual que en la anterior técnica, se usa un subconjunto del conjunto de entrenamiento debido a la complejidad computacional. Al final se representa el resultado en 2D para analizar la agrupación de las clases tras la transformación.

In [ ]:
# ISOMAP
X = Tables.table(Xtr_pca30_small)
isomap_model = MyIsomap(n_components=2, k=12)
mach_isomap = machine(isomap_model, X)
fit!(mach_isomap)
Xtrain_isomap = MMI.transform(mach_isomap, X)

println("Isomap hecho")
plot_2d_projection(Xtrain_isomap, y_small;
                   title_str="Isomap")

## 6. LLE (Locally Linear Embedding)
Aquí se utiliza LLE para obtener una proyección en dos dimensiones a partir de las relaciones locales entre los datos. Para mejorar la estabilidad del modelo se aplica primero PCA a 10 dimensiones y, tras reducir el conjunto resultante, LLE se aplica sobre ese resultado. La proyección final se muestra en 2D para ver cómo quedan distribuidas las clases en este espacio reducido.

In [ ]:
pca10 = PCA(maxoutdim=10)
mach_pca10 = machine(pca10, Xtrain_scaled)
fit!(mach_pca10)

Xtr_pca10 = MMI.transform(mach_pca10, Xtrain_scaled)
Xtr_pca10_mat = MMI.matrix(Xtr_pca10)

# Submuestreo
n = size(Xtr_pca10_mat, 1)
idx = randperm(n)[1:2000]
Xtr_pca10_small = Xtr_pca10_mat[idx, :]
y_small = y_trainval[idx]

# convertir features PCA a DataFrame
X_df = DataFrame(Xtr_pca10_small, :auto)

# guardar IDs por separado
ids_original = collect(1:size(X_df, 1))

# entrenar LLE solo con features
lle_model = MyLLE(n_components=2, k=40)
mach_lle = machine(lle_model, X_df)
fit!(mach_lle)

# transformar
Xtrain_lle_tbl = MMI.transform(mach_lle, X_df)

# convertir salida a DataFrame
Xtrain_lle_df = DataFrame(Xtrain_lle_tbl)

# convertir a matriz para plot
Xtrain_lle_mat = Matrix(Xtrain_lle_df)

# ajustar etiquetas
n_lle = size(Xtrain_lle_mat, 1)
y_lle = y_small[1:n_lle]

println("LLE hecho")

plot_2d_projection(
    Xtrain_lle_mat,
    y_lle;
    title_str="LLE (train subset)"
)


## 7. Análisis de separabilidad por método

Calculamos la separabilidad de clases usando el ratio de dispersión inter-clase vs intra-clase, obteniendo una tabla con los resultados finales

In [ ]:
methods = [
    ("PCA", Xtrain_pca,          y_trainval),
    ("LDA", Xtrain_lda,          y_trainval),
    ("ICA", Xtrain_ica,          y_trainval),
    ("t-SNE", tsne_output,       y_small),
    ("Isomap", Xtrain_isomap,    y_small),
    ("LLE", Xtrain_lle_tbl,      y_lle)
]

results = DataFrame(
    Method = String[],
    Separability = Float64[]
)

for (name, X_2d, y) in methods
    X = to_matrix_2d(X_2d)
    sep = calculate_separability(X, y)
    push!(results, (name, sep))
end

sort!(results, :Separability, rev=true)
println("\nRanking de separabilidad de clases:")
pretty_table(results)

## 8. Conclusiones
Los resultados muestran diferencias claras en la capacidad de cada técnica para separar las clases en un espacio bidimensional:

1. LDA obtiene la mejor separabilidad con mucha diferencia:
Es la única técnica supervisada del conjunto, y aprovecha directamente las etiquetas durante el entrenamiento. Su objetivo consiste precisamente en maximizar la distancia entre clases, por lo que es lógico que lidere la tabla con un valor muy superior al resto.

2. Isomap ofrece la mejor separación entre los métodos no supervisados:
Preserva distancias geodésicas, lo que permite capturar relaciones globales no lineales en los datos. Esto resulta especialmente útil en un dataset complejo como el de actividad humana.

3. PCA presenta una buena separación, mejor que t-SNE e ICA:
Aunque es lineal, logra retener suficiente estructura del espacio original, lo que indica que parte de la variabilidad relevante está alineada con direcciones principales del espacio.

4. t-SNE muestra una separabilidad moderada:
Visualmente, tiende a generar clusters muy llamativos, pero su objetivo no es maximizar distancias entre clases, lo que explica el valor relativamente bajo en esta métrica.

5. ICA y LLE obtienen los valores más bajos de separabilidad.
        ICA busca independencia estadística entre componentes, no separación por clases.
        LLE es sensible al ruido y se centra en la reconstrucción local, no global.
   En ambos casos, esto se traduce en una proyección menos útil para discriminar actividades.